# Kaggle Smoke Debug

Run this notebook first to verify repo checkout, HF token, dataset parsing, feature cache, single-GPU smoke, 2-GPU DDP smoke, evaluation, benchmark, and debug artifacts.

In [ ]:
GITHUB_REPO_URL = "https://github.com/<your-user>/<your-repo>.git"
GITHUB_BRANCH = "huy"

import os
os.environ["GITHUB_REPO_URL"] = GITHUB_REPO_URL
os.environ["GITHUB_BRANCH"] = GITHUB_BRANCH
print({"repo": GITHUB_REPO_URL, "branch": GITHUB_BRANCH})

In [ ]:
%%bash
set -e
cd /kaggle/working
if [ ! -d Efficient_VLM_For_Autonomous_Driving ]; then
  git clone --branch "$GITHUB_BRANCH" "$GITHUB_REPO_URL" Efficient_VLM_For_Autonomous_Driving
fi
cd Efficient_VLM_For_Autonomous_Driving
git pull --ff-only
python -m pip install -e .

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print({"hf_token_present": bool(os.environ.get("HF_TOKEN"))})

In [ ]:
import torch
print({"cuda": torch.cuda.is_available(), "gpu_count": torch.cuda.device_count()})
for idx in range(torch.cuda.device_count()):
    print(idx, torch.cuda.get_device_name(idx))

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG=configs/repvit_t5_efficient_tiny_smoke.yaml
python -m efficient_vlm_ad inspect-data --config "$CONFIG" --debug --debug-samples 3
python -m efficient_vlm_ad prepare-data --config "$CONFIG" --subset smoke --debug --debug-samples 3
python -m efficient_vlm_ad debug-sample --config "$CONFIG" --split train --index 0 --debug --debug-samples 3
python -m efficient_vlm_ad prepare-features --config "$CONFIG" --subset smoke --debug --debug-samples 3

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG=configs/repvit_t5_efficient_tiny_smoke.yaml
python -m efficient_vlm_ad train --config "$CONFIG" --stage align --max-steps 20 --debug --debug-samples 3
python -m efficient_vlm_ad train --config "$CONFIG" --stage finetune --resume outputs/repvit_t5_efficient_tiny_smoke/checkpoints/align_latest.pt --max-steps 20 --debug --debug-samples 3
python -m efficient_vlm_ad evaluate --config "$CONFIG" --checkpoint outputs/repvit_t5_efficient_tiny_smoke/checkpoints/finetune_latest.pt --max-samples 32 --debug --debug-samples 3
python -m efficient_vlm_ad benchmark --config "$CONFIG" --checkpoint outputs/repvit_t5_efficient_tiny_smoke/checkpoints/finetune_latest.pt --max-samples 32 --debug --debug-samples 3

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG=configs/repvit_t5_efficient_tiny_smoke.yaml
accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision fp16 --dynamo_backend no -m efficient_vlm_ad train --config "$CONFIG" --stage align --max-steps 20 --debug --debug-samples 1

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
PROFILE=outputs/repvit_t5_efficient_tiny_smoke
ls -lh "$PROFILE/checkpoints" || true
tail -n 20 "$PROFILE/debug/debug_events.jsonl" || true
cat "$PROFILE/metrics.json" || true
cat "$PROFILE/benchmark.json" || true